In [2]:
import sys
sys.path.append("..")

import pandas as pd
from src.db import get_engine
from src.job_titles import clean_title, job_family

titles = pd.read_sql(
    "SELECT TRY_CAST(id AS bigint) AS id, emp_title FROM dbo.loans_population",
    get_engine(),
)

splits = {}
for name in ["train", "valid", "test"]:
    df = pd.read_parquet(f"../data/processed/{name}.parquet")
    df = df.merge(titles, on="id", how="left")
    df["job_family"] = df["emp_title"].apply(job_family)
    splits[name] = df

train = splits["train"]
print({name: len(df) for name, df in splits.items()})

{'train': 335558, 'valid': 120790, 'test': 162236}


In [3]:
coverage = train["job_family"].value_counts(normalize=True).mul(100).round(1)
print("Share of training loans by job family (%):")
coverage

Share of training loans by job family (%):


job_family
Other                          41.3
Management                     10.3
Missing                         6.7
Office & admin                  5.4
Engineering & IT                4.7
Healthcare                      4.7
Legal, finance & consulting     4.3
Skilled trades                  4.3
Senior management               3.7
Education                       3.4
Nursing                         2.5
Sales & retail                  2.4
Driving & transport             2.1
Public safety & military        1.8
Food & hospitality              1.0
Labor & service                 0.8
Owner & self-employed           0.7
Name: proportion, dtype: float64

In [4]:
family_table = train.groupby("job_family").agg(
    loans=("target", "size"),
    default_pct=("target", "mean"),
    median_income=("annual_inc", "median"),
)
family_table["default_pct"] = (family_table["default_pct"] * 100).round(1)
family_table["share_pct"] = (family_table["loans"] / len(train) * 100).round(1)
family_table.sort_values("default_pct")

,loans,default_pct,median_income,share_pct
job_family,,,,
Engineering & IT,15677,8.7,80000.0,4.7
Senior management,12478,8.8,107000.0,3.7
Public safety & military,6147,10.1,70000.0,1.8
"Legal, finance & consulting",14410,10.6,65000.0,4.3
Education,11481,11.2,60000.0,3.4
Healthcare,15619,12.5,60800.0,4.7
Management,34458,12.8,66000.0,10.3
Other,138499,12.8,59500.0,41.3
Nursing,8260,13.3,70000.0,2.5


In [5]:
other = train.loc[train["job_family"] == "Other", "emp_title"].map(clean_title)
other.value_counts().head(40)

emp_title
at t                     312
officer                  203
state of california      190
counselor                184
management               176
team leader              176
department of defense    175
buyer                    172
ibm                      157
pastor                   156
walgreens                155
lockheed martin          141
jp morgan chase          139
inspector                138
captain                  135
target                   133
production               133
verizon wireless         131
jpmorgan chase           128
assembler                128
graphic designer         122
loan officer             122
designer                 121
verizon                  120
recruiter                120
united parcel service    119
conductor                119
realtor                  118
marketing                108
accounts payable         106
best buy                 105
correction officer       105
comcast                  104
investigator             101
esti

In [6]:
from src.job_titles import JOB_FAMILIES, job_family

print([family for family, _ in JOB_FAMILIES])
print(job_family("usps"), "|", job_family("laborer"), "|", job_family("USAF"))

['Nursing', 'Healthcare', 'Education', 'Public safety & military', 'Senior management', 'Legal, finance & consulting', 'Engineering & IT', 'Driving & transport', 'Skilled trades', 'Food & hospitality', 'Management', 'Labor & service', 'Sales & retail', 'Office & admin', 'Owner & self-employed']
Driving & transport | Labor & service | Public safety & military


In [7]:
import lightgbm
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve, brier_score_loss
from sklearn.pipeline import Pipeline

from src.features import build_features, get_numeric_features, CATEGORICAL_FEATURES, BINARY_FEATURES
from src.models import build_lightgbm_preprocessor, build_lightgbm_model, CalibratedModel, logit

X = {name: df.drop(columns=["target"]) for name, df in splits.items()}
y = {name: df["target"] for name, df in splits.items()}

numeric_features = get_numeric_features(
    build_features(X["train"].drop(columns=["emp_title", "job_family"]).head(1000))
)

CHOSEN_PARAMS = {
    "num_leaves": 15,
    "min_child_samples": 500,
    "cat_smooth": 100,
    "min_data_per_group": 1000,
}
print("Numeric features:", len(numeric_features))

Numeric features: 57


In [8]:
def train_lightgbm(categorical):
    prep = build_lightgbm_preprocessor(numeric_features, categorical, BINARY_FEATURES)
    X_tr = prep.fit_transform(X["train"])
    X_va = prep.transform(X["valid"])

    model = build_lightgbm_model(**CHOSEN_PARAMS)
    model.fit(
        X_tr, y["train"],
        eval_X=X_va,
        eval_y=y["valid"],
        eval_metric="auc",
        categorical_feature=categorical,
        callbacks=[lightgbm.early_stopping(stopping_rounds=100, verbose=False)],
    )
    return Pipeline([("prep", prep), ("model", model)])


def calibrate(pipeline):
    raw_valid = pipeline.predict_proba(X["valid"])[:, 1]
    calibrator = LogisticRegression()
    calibrator.fit(logit(raw_valid).reshape(-1, 1), y["valid"])
    return CalibratedModel(pipeline, calibrator)


with_job = CATEGORICAL_FEATURES + ["job_family"]

models = {
    "Without job family": calibrate(train_lightgbm(CATEGORICAL_FEATURES)),
    "With job family": calibrate(train_lightgbm(with_job)),
}

p = {
    name: {split: m.predict_proba(X[split])[:, 1] for split in ["valid", "test"]}
    for name, m in models.items()
}

for name, m in models.items():
    trees = m.pipeline.named_steps["model"].best_iteration_
    print(f"{name}: {trees} trees, validation AUC {roc_auc_score(y['valid'], p[name]['valid']):.4f}")

Without job family: 813 trees, validation AUC 0.6951
With job family: 941 trees, validation AUC 0.6975


In [9]:
def ks_statistic(y_true, scores):
    fpr, tpr, _ = roc_curve(y_true, scores)
    return (tpr - fpr).max()


test_table = pd.DataFrame({
    name: {
        "auc": roc_auc_score(y["test"], preds["test"]),
        "ks": ks_statistic(y["test"], preds["test"]),
        "brier": brier_score_loss(y["test"], preds["test"]),
        "avg_predicted": preds["test"].mean(),
    }
    for name, preds in p.items()
}).T
test_table["gini"] = 2 * test_table["auc"] - 1
print(test_table.round(4))

rng = np.random.default_rng(42)
y_arr = y["test"].values
a, b = p["With job family"]["test"], p["Without job family"]["test"]
differences = []
for _ in range(200):
    s = rng.integers(0, len(y_arr), size=len(y_arr))
    differences.append(roc_auc_score(y_arr[s], a[s]) - roc_auc_score(y_arr[s], b[s]))
differences = np.array(differences)
low, high = np.percentile(differences, [2.5, 97.5])
print(f"\nAUC gain from job family: {differences.mean():.4f} (95% range {low:.4f} to {high:.4f})")
print(f"With job family ahead in {(differences > 0).mean() * 100:.0f}% of samples")

                       auc      ks   brier  avg_predicted    gini
Without job family  0.7026  0.2941  0.1170         0.1462  0.4051
With job family     0.7044  0.2962  0.1168         0.1461  0.4087

AUC gain from job family: 0.0018 (95% range 0.0009 to 0.0027)
With job family ahead in 100% of samples


In [10]:
profit = {s: (splits[s]["total_pymnt"] - splits[s]["funded_amnt"]).values for s in ["valid", "test"]}
thresholds = np.round(np.arange(0.05, 0.605, 0.005), 3)

money = {}
for name, preds in p.items():
    totals = [profit["valid"][preds["valid"] < t].sum() for t in thresholds]
    cutoff = thresholds[int(np.argmax(totals))]
    approved = preds["test"] < cutoff
    money[name] = {
        "cutoff": cutoff,
        "approval_rate_pct": approved.mean() * 100,
        "gain_vs_approve_all": (profit["test"][approved].sum() - profit["test"].sum()) / 1e6,
    }
pd.DataFrame(money).T.round(3)

,cutoff,approval_rate_pct,gain_vs_approve_all
Without job family,0.36,97.183,2.552
With job family,0.36,96.998,3.049


In [11]:
model_with = models["With job family"].pipeline.named_steps["model"]
importance = pd.DataFrame({
    "feature": model_with.feature_name_,
    "gain": model_with.booster_.feature_importance(importance_type="gain"),
})
importance["share_pct"] = (importance["gain"] / importance["gain"].sum() * 100).round(1)
importance = importance.sort_values("gain", ascending=False).reset_index(drop=True)
importance.index += 1

print("Rank of job_family:", importance.index[importance["feature"] == "job_family"][0])
importance.head(12)

Rank of job_family: 5


,feature,gain,share_pct
1,fico_avg,38148.156328,13.4
2,addr_state,25569.843030,9.0
3,annual_inc,16927.003212,5.9
4,loan_to_income,15925.043402,5.6
5,job_family,14290.727194,5.0
6,acc_open_past_24mths,12662.843221,4.4
7,dti,10472.875017,3.7
8,purpose,9894.521526,3.5
9,bc_open_to_buy,8639.521372,3.0
10,inq_last_6mths,6487.247323,2.3


In [12]:
all_loans = pd.concat([splits["train"], splits["valid"], splits["test"]])
by_year = all_loans.groupby(all_loans["issue_date"].dt.year).agg(
    loans=("target", "size"),
    missing_pct=("job_family", lambda s: (s == "Missing").mean() * 100),
    other_pct=("job_family", lambda s: (s == "Other").mean() * 100),
)
print(by_year.round(1))

from src.monitoring import psi_categorical
print("\nPSI of job_family, test vs train:",
      round(psi_categorical(splits["train"]["job_family"], splits["test"]["job_family"]), 3))

             loans  missing_pct  other_pct
issue_date                                
2007           251          9.6       74.1
2008          1562          4.4       76.0
2009          4716          5.7       75.6
2010          8466          6.4       74.6
2011         14101          7.2       74.5
2012         43470          6.4       73.2
2013        100422          7.1       55.9
2014        162570          6.5       17.7
2015        283026          6.7       18.0

PSI of job_family, test vs train: 0.325


In [13]:
def group_check(column, split="test", min_loans=1000):
    df = splits[split]
    rows = []
    for name, preds in p.items():
        data = pd.DataFrame({"group": df[column].values, "actual": y[split].values, "predicted": preds[split]})
        g = data.groupby("group", observed=True).agg(
            loans=("actual", "size"), actual=("actual", "mean"), predicted=("predicted", "mean"),
        )
        g = g[g["loans"] >= min_loans]
        g["gap_pts"] = (g["predicted"] - g["actual"]) * 100
        g["model"] = name
        rows.append(g)
    table = pd.concat(rows).reset_index()
    return table.pivot(index="group", columns="model", values="gap_pts").round(2)


job_gaps = group_check("job_family")
print("Average absolute gap:\n", job_gaps.abs().mean().round(2))
job_gaps

Average absolute gap:
 model
With job family       0.58
Without job family    1.54
dtype: float64


model,With job family,Without job family
group,,
Driving & transport,-1.68,-4.21
Education,-0.18,0.55
Engineering & IT,0.19,2.20
Food & hospitality,0.77,-2.27
Healthcare,0.86,0.75
Labor & service,0.19,-1.63
"Legal, finance & consulting",0.31,1.85
Management,0.15,-0.01
Missing,-1.32,-1.55


In [14]:
splits["test"]["income_band"] = pd.qcut(
    splits["test"]["annual_inc"], q=4,
    labels=["Lowest 25%", "Lower-middle", "Upper-middle", "Highest 25%"],
)

for column in ["income_band", "home_ownership", "verification_status", "addr_state"]:
    gaps = group_check(column)
    print(f"\n{column}: average absolute gap")
    print(gaps.abs().mean().round(2).to_string())

low_income = (splits["test"]["income_band"] == "Lowest 25%").values
for name, preds in p.items():
    auc = roc_auc_score(y["test"][low_income], preds["test"][low_income])
    print(f"\nLowest-income AUC, {name}: {auc:.3f}")


income_band: average absolute gap
model
With job family       0.35
Without job family    0.37

home_ownership: average absolute gap
model
With job family       0.42
Without job family    0.39

verification_status: average absolute gap
model
With job family       0.51
Without job family    0.46

addr_state: average absolute gap
model
With job family       1.01
Without job family    1.03

Lowest-income AUC, Without job family: 0.673

Lowest-income AUC, With job family: 0.672


In [15]:
cutoff = 0.36
approval = pd.DataFrame({
    name: pd.Series(preds["test"] < cutoff).groupby(splits["test"]["job_family"].values).mean() * 100
    for name, preds in p.items()
})
approval["change_pts"] = approval["With job family"] - approval["Without job family"]
approval.round(1).sort_values("change_pts")

,Without job family,With job family,change_pts
Food & hospitality,96.3,93.2,-3.1
Driving & transport,96.2,93.9,-2.3
Labor & service,96.3,94.0,-2.3
Sales & retail,97.2,95.6,-1.6
Owner & self-employed,98.5,97.0,-1.4
Skilled trades,96.9,95.9,-1.0
Nursing,97.5,97.0,-0.5
Other,97.1,96.7,-0.4
Healthcare,97.7,97.4,-0.3
Missing,92.6,92.5,-0.2


In [17]:
for year in [2010, 2012, 2014]:
    titles_that_year = all_loans.loc[all_loans["issue_date"].dt.year == year, "emp_title"].dropna()
    top = titles_that_year.str.strip().str.lower().value_counts().head(12).index.tolist()
    print(f"\n{year}: {top}")


2010: ['us army', 'bank of america', 'walmart', 'at&t', 'wells fargo', 'ibm', 'verizon wireless', 'usps', 'lockheed martin', 'walgreens', 'jp morgan chase', 'fidelity investments']

2012: ['us army', 'bank of america', 'walmart', 'kaiser permanente', 'at&t', 'usps', 'wells fargo', 'ups', 'usaf', 'us postal service', 'united states postal service', 'united states air force']

2014: ['teacher', 'manager', 'registered nurse', 'owner', 'supervisor', 'sales', 'driver', 'rn', 'office manager', 'project manager', 'general manager', 'truck driver']


In [18]:
recent = (X["train"]["issue_date"] >= "2013-01-01").values


def train_recent(categorical):
    prep = build_lightgbm_preprocessor(numeric_features, categorical, BINARY_FEATURES)
    X_tr = prep.fit_transform(X["train"][recent])
    X_va = prep.transform(X["valid"])

    model = build_lightgbm_model(**CHOSEN_PARAMS)
    model.fit(
        X_tr, y["train"][recent],
        eval_X=X_va,
        eval_y=y["valid"],
        eval_metric="auc",
        categorical_feature=categorical,
        callbacks=[lightgbm.early_stopping(stopping_rounds=100, verbose=False)],
    )
    return calibrate(Pipeline([("prep", prep), ("model", model)]))


recent_models = {
    "2013-14 only, without job family": train_recent(CATEGORICAL_FEATURES),
    "2013-14 only, with job family": train_recent(with_job),
}

print("Training loans used:", int(recent.sum()))
summary = {}
for name, m in recent_models.items():
    p_test = m.predict_proba(X["test"])[:, 1]
    summary[name] = {
        "test_auc": roc_auc_score(y["test"], p_test),
        "avg_predicted": p_test.mean(),
    }
pd.DataFrame(summary).T.round(4)

Training loans used: 262992


,test_auc,avg_predicted
"2013-14 only, without job family",0.7012,0.1464
"2013-14 only, with job family",0.7046,0.1464


In [19]:
all_models = {**models, **recent_models}

final = {}
for name, m in all_models.items():
    p_valid = m.predict_proba(X["valid"])[:, 1]
    p_test = m.predict_proba(X["test"])[:, 1]
    totals = [profit["valid"][p_valid < t].sum() for t in thresholds]
    cutoff = thresholds[int(np.argmax(totals))]
    approved = p_test < cutoff
    final[name] = {
        "test_auc": roc_auc_score(y["test"], p_test),
        "cutoff": cutoff,
        "approval_rate_pct": approved.mean() * 100,
        "gain_vs_approve_all": (profit["test"][approved].sum() - profit["test"].sum()) / 1e6,
    }

final_table = pd.DataFrame(final).T
final_table.round(4)

,test_auc,cutoff,approval_rate_pct,gain_vs_approve_all
Without job family,0.7026,0.36,97.1831,2.5517
With job family,0.7044,0.36,96.9976,3.0488
"2013-14 only, without job family",0.7012,0.34,96.1137,2.8277
"2013-14 only, with job family",0.7046,0.34,95.9522,3.2565
